In [ ]:
import numpy as np
from scipy.sparse import csr_matrix
from sklearn.preprocessing import normalize

In [ ]:
class ItemBasedKNNModel:
    def __init__(self, k=20):
        self.k = k
        self.similarity_matrix = None
        self.user_item_matrix = None
        
        # Diccionarios para traducir IDs reales a índices de la matriz y viceversa:
        self.user_id_to_idx = {}
        self.item_id_to_idx = {}
        self.idx_to_item_id = {}
        
    def fit(self, df, user_col, item_col, score_col):
        """
        Construye la matriz usuario-ítem y calcula la matriz de similitud de ítems.
        """
        print("Mapeando IDs a índices...")
        unique_users = df[user_col].unique()
        unique_items = df[item_col].unique()
        
        self.user_id_to_idx = {user: idx for idx, user in enumerate(unique_users)}
        self.item_id_to_idx = {item: idx for idx, item in enumerate(unique_items)}
        self.idx_to_item_id = {idx: item for item, idx in self.item_id_to_idx.items()}
        
        # Convertimos las columnas a sus índices correspondientes:
        user_indices = df[user_col].map(self.user_id_to_idx).values
        item_indices = df[item_col].map(self.item_id_to_idx).values
        scores = df[score_col].values
        
        print("Construyendo matriz dispersa (Sparse Matrix)...")
        # Matriz R dispersa (usando CSR de scipy) con filas = usuarios y columnas = ítems: 
        self.user_item_matrix = csr_matrix(
            (scores, (user_indices, item_indices)), 
            shape=(len(unique_users), len(unique_items))
        )
        
        print("Calculando similitud del coseno vectorizada...")
        # Normalizamos las columnas (ítems) para la similitud del coseno (norma L2):
        R_normalized = normalize(self.user_item_matrix, norm='l2', axis=0)
        
        # S = R^T * R (Multiplicación de matrices dispersas).
        # Esto calcula el producto punto entre todos los ítems normalizados (similitud coseno):
        item_similarity_full = R_normalized.T.dot(R_normalized)
        
        print(f"Aplicando restricción de K vecinos (K={self.k})...")
        # Conservamos solo los K valores más altos por fila.
        # Convertimos a formato LIL para modificar la estructura eficientemente: 
        item_similarity_lil = item_similarity_full.tolil()
        
        # Iteramos sobre las filas (ítems) para quedarnos solo con el Top-K: 
        for i in range(item_similarity_lil.shape[0]):
            # Ignoramos la similitud de un ítem consigo mismo (diagonal): 
            item_similarity_lil[i, i] = 0
            
            row = item_similarity_lil.data[i]
            if len(row) > self.k:
                # Encontramos el umbral del K-ésimo valor más grande: 
                threshold = np.sort(row)[-self.k]
                # Ponemos a 0 los valores por debajo del umbral:
                item_similarity_lil[i, item_similarity_full[i].toarray()[0] < threshold] = 0
                
        # Volvemos a formato CSR para operaciones matemáticas rápidas en la predicción:
        self.similarity_matrix = item_similarity_lil.tocsr()
        print("Modelo KNN entrenado con éxito.")

    def recommend(self, user_id, n=10):
        """
        Genera el Top-N de recomendaciones para un usuario específico.
        """
        if user_id not in self.user_id_to_idx:
            return [] # Usuario desconocido (Cold Start).
            
        u_idx = self.user_id_to_idx[user_id]
        
        # Obtenemos el vector historial del usuario:
        user_vector = self.user_item_matrix[u_idx]
        
        # Predicción: p_u = r_u * S
        # Multiplicamos el vector del usuario por la matriz de similitud:
        user_predictions = user_vector.dot(self.similarity_matrix).toarray()[0]
        
        # Filtramos las canciones que el usuario ya ha escuchado (no queremos recomendarlas de nuevo):
        interacted_items_indices = user_vector.indices
        user_predictions[interacted_items_indices] = -1 # Asignamos un score negativo.
        
        # Obtenemos los índices de las N canciones con mayor predicción:
        top_n_indices = user_predictions.argsort()[-n:][::-1]
        
        # Traducimos de vuelta a IDs reales:
        top_n_items = [self.idx_to_item_id[idx] for idx in top_n_indices]
        return top_n_items